In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
from google.colab import files

uploaded = files.upload()

Saving fer dataset.zip to fer dataset.zip


In [2]:
!unzip -q "fer dataset.zip" -d /content/fer_dataset

In [3]:
!find /content/fer_dataset -maxdepth 2 -type d

/content/fer_dataset
/content/fer_dataset/test
/content/fer_dataset/test/happy
/content/fer_dataset/test/fear
/content/fer_dataset/test/surprise
/content/fer_dataset/test/sad
/content/fer_dataset/test/disgust
/content/fer_dataset/test/angry
/content/fer_dataset/test/neutral
/content/fer_dataset/train
/content/fer_dataset/train/happy
/content/fer_dataset/train/fear
/content/fer_dataset/train/surprise
/content/fer_dataset/train/sad
/content/fer_dataset/train/disgust
/content/fer_dataset/train/angry
/content/fer_dataset/train/neutral


In [20]:
import os
import numpy as np
import cv2
from sklearn.utils.class_weight import compute_class_weight

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

EMOTIONS = [
    'angry',
    'disgust',
    'fear',
    'happy',
    'neutral',
    'sad',
    'surprise'
]

IMG_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 100

TRAIN_DIR = '/content/fer_dataset/train'
TEST_DIR = '/content/fer_dataset/test'

In [21]:
def load_images_from_folder(base_dir):

    images = []
    labels = []

    for label_idx, emotion in enumerate(EMOTIONS):

        folder = os.path.join(base_dir, emotion)

        if not os.path.exists(folder):
            folder = os.path.join(base_dir, emotion.capitalize())

        if not os.path.exists(folder):
            continue

        files = [
            f for f in os.listdir(folder)
            if f.lower().endswith(
                ('.jpg', '.jpeg', '.png')
            )
        ]

        print(f"{emotion}: {len(files)}")

        for fname in files:

            path = os.path.join(folder, fname)

            img = cv2.imread(
                path,
                cv2.IMREAD_GRAYSCALE
            )

            if img is None:
                continue

            img = cv2.resize(
                img,
                (IMG_SIZE, IMG_SIZE)
            )

            img = img.reshape(
                IMG_SIZE,
                IMG_SIZE,
                1
            )

            images.append(img / 255.0)
            labels.append(label_idx)

    return (
        np.array(images, dtype=np.float32),
        np.array(labels, dtype=np.int32)
    )

In [22]:
def augment(image, label):

    image = tf.image.random_flip_left_right(
        image
    )

    image = tf.image.random_brightness(
        image,
        max_delta=0.3
    )

    image = tf.image.random_contrast(
        image,
        lower=0.7,
        upper=1.3
    )

    return image, label

In [23]:
def residual_block(
    x,
    filters,
    stride=1
):

    shortcut = x

    x = layers.Conv2D(
        filters,
        3,
        strides=stride,
        padding='same',
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(
        filters,
        3,
        padding='same',
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)

    if (
        stride != 1 or
        shortcut.shape[-1] != filters
    ):

        shortcut = layers.Conv2D(
            filters,
            1,
            strides=stride,
            padding='same',
            use_bias=False
        )(shortcut)

        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.add([x, shortcut])

    x = layers.Activation('relu')(x)

    return x

In [24]:
def build_resnet_model():

    inputs = keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 1)
    )

    x = layers.Conv2D(
        64,
        3,
        padding='same',
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = residual_block(x, 64)
    x = layers.Dropout(0.2)(x)

    x = residual_block(
        x,
        128,
        stride=2
    )

    x = residual_block(x, 128)
    x = layers.Dropout(0.25)(x)

    x = residual_block(
        x,
        256,
        stride=2
    )

    x = residual_block(x, 256)
    x = layers.Dropout(0.3)(x)

    x = residual_block(
        x,
        512,
        stride=2
    )

    x = layers.Dropout(0.4)(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        256,
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(
        len(EMOTIONS),
        activation='softmax'
    )(x)

    model = keras.Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            3e-4
        ),
        loss=tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=0.1
        ),
        metrics=['accuracy']
    )

    return model

In [25]:
print("Loading training data...")
X_train, y_train = load_images_from_folder(
    TRAIN_DIR
)

print("\nLoading test data...")
X_test, y_test = load_images_from_folder(
    TEST_DIR
)

print(X_train.shape)
print(X_test.shape)

Loading training data...
angry: 3995
disgust: 436
fear: 4097
happy: 7215
neutral: 4965
sad: 4830
surprise: 3171

Loading test data...
angry: 958
disgust: 111
fear: 1024
happy: 1774
neutral: 1233
sad: 1247
surprise: 831
(28709, 64, 64, 1)
(7178, 64, 64, 1)


In [26]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(
    enumerate(class_weights)
)

print(class_weight_dict)

{0: np.float64(1.0266046844269623), 1: np.float64(9.406618610747051), 2: np.float64(1.0010460615781582), 3: np.float64(0.5684387684387684), 4: np.float64(0.8260394187886635), 5: np.float64(0.8491274770777877), 6: np.float64(1.293372978330405)}


In [27]:
val_size = int(
    len(X_train) * 0.1
)

indices = np.random.permutation(
    len(X_train)
)

val_idx = indices[:val_size]
train_idx = indices[val_size:]

X_val = X_train[val_idx]
y_val = y_train[val_idx]

X_tr = X_train[train_idx]
y_tr = y_train[train_idx]

In [28]:
y_tr_cat = keras.utils.to_categorical(
    y_tr,
    len(EMOTIONS)
)

y_val_cat = keras.utils.to_categorical(
    y_val,
    len(EMOTIONS)
)

y_test_cat = keras.utils.to_categorical(
    y_test,
    len(EMOTIONS)
)

In [29]:
train_ds = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_tr, y_tr_cat)
    )
    .shuffle(len(X_tr))
    .map(
        augment,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_val, y_val_cat)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_test, y_test_cat)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [30]:
model = build_resnet_model()

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 64, 64, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        576 │ input_layer_2[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64, 64,    │          0 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     73,728 │ dropout_2[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    147,456 │ activation_3[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │      8,192 │ dropout_2[0][0] 

 Total params: 6,514,503 (24.85 MB)

 Trainable params: 6,506,695 (24.82 MB)

 Non-trainable params: 7,808 (30.50 KB)

In [31]:
os.makedirs(
    'models',
    exist_ok=True
)

cbs = [

    callbacks.ModelCheckpoint(
        'models/fer_model.keras',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    ),

    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        patience=4,
        factor=0.5,
        min_lr=1e-6,
        verbose=1
    ),

    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

In [32]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=cbs
)

Epoch 1/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step - accuracy: 0.1385 - loss: 2.3870
Epoch 1: val_accuracy improved from None to 0.24564, saving model to models/fer_model.keras

Epoch 1: finished saving model to models/fer_model.keras
404/404 ━━━━━━━━━━━━━━━━━━━━ 112s 202ms/step - accuracy: 0.1419 - loss: 2.3104 - val_accuracy: 0.2456 - val_loss: 2.0246 - learning_rate: 3.0000e-04
Epoch 2/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - accuracy: 0.1531 - loss: 2.1683
Epoch 2: val_accuracy did not improve from 0.24564
404/404 ━━━━━━━━━━━━━━━━━━━━ 59s 147ms/step - accuracy: 0.1524 - loss: 2.1722 - val_accuracy: 0.1854 - val_loss: 1.9404 - learning_rate: 3.0000e-04
Epoch 3/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.1596 - loss: 2.1200
Epoch 3: val_accuracy did not improve from 0.24564
404/404 ━━━━━━━━━━━━━━━━━━━━ 59s 145ms/step - accuracy: 0.1663 - loss: 2.0793 - val_accuracy: 0.2063 - val_loss: 1.8909 - learning_rate: 3.0000e-04
Epoch 4/100
404/404 ━━━━━━━━━━━━━━━━

In [35]:
print(model.input_shape)
print(model.output_shape)
model.summary()

(None, 64, 64, 1)
(None, 7)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 64, 64, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        576 │ input_layer_2[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     36,864 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64, 64,    │          0 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     73,728 │ dropout_2[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    147,456 │ activation_3[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │      8,192 │ dropout_2[0][0] 

 Total params: 19,527,895 (74.49 MB)

 Trainable params: 6,506,695 (24.82 MB)

 Non-trainable params: 7,808 (30.50 KB)

 Optimizer params: 13,013,392 (49.64 MB)

In [33]:
loss, acc = model.evaluate(
    test_ds,
    verbose=0
)

print(
    f"Test Accuracy: {acc*100:.2f}%"
)

Test Accuracy: 64.15%


In [34]:
from google.colab import files

files.download(
    'models/fer_model.keras'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>